# Chapter 6 &mdash; DeMorgan's Law for DFA, Verified by Isomorphism

**Concept 12 of the Chapter 6 decomposition:** *DeMorgan's Law for DFA, Verified by Isomorphism*

$L_1\cap L_2 = \overline{\overline{L_1}\cup\overline{L_2}}$ &mdash; confirmed by minimizing both sides.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-DeMorgan-For-DFA/Concept-DeMorgan-For-DFA.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


DeMorgan's law holds for languages:
$$L_1 \cap L_2 = \overline{\overline{L_1} \cup \overline{L_2}}.$$

So intersection is **redundant** as a primitive: complement and union suffice. This is
the same closure argument you will use for regular expressions in Chapter 10.

And it is **checkable**. Build both sides, minimize both, and ask `iso_dfa`. By
Myhill&ndash;Nerode a `True` answer is a proof for these particular languages &mdash; the
strongest machine-checked evidence available without a general proof.

## 2. Definitions

### Two languages

In [ ]:
even0 = md2mc('''DFA
IF : 0 -> Od
IF : 1 -> IF
Od : 0 -> IF
Od : 1 -> Od
''')
even1 = md2mc('''DFA
IF : 1 -> Od
IF : 0 -> IF
Od : 1 -> IF
Od : 0 -> Od
''')

### Both sides of DeMorgan's law

In [ ]:
def lhs(A, B): return min_dfa(pruneUnreach(intersect_dfa(A, B)))
def rhs(A, B): return min_dfa(pruneUnreach(
                   comp_dfa(union_dfa(comp_dfa(A), comp_dfa(B)))))

## 3. Tests

The two sides minimize to the same size.

In [ ]:
L, R = lhs(even0, even1), rhs(even0, even1)
print("LHS (intersection)     : %d states" % len(L["Q"]))
print("RHS (DeMorgan version) : %d states" % len(R["Q"]))
assert len(L["Q"]) == len(R["Q"])

And they are **isomorphic** &mdash; which by Myhill&ndash;Nerode means identical languages.

In [ ]:
print("langeq_dfa :", langeq_dfa(L, R))
print("iso_dfa    :", iso_dfa(L, R))
assert langeq_dfa(L, R) and iso_dfa(L, R)
print("\nMinimal + isomorphic  =>  the same language, proved by the theorem.")

Cross-checked against the arithmetic specification, both ways.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(11) for p in product('01', repeat=k)]
spec = lambda s: s.count('0') % 2 == 0 and s.count('1') % 2 == 0
for name, X in [('LHS', L), ('RHS', R)]:
    assert all(accepts_dfa(X, s) == spec(s) for s in strs)
    print("%s matches the spec on all %d strings" % (name, len(strs)))

The dual law too: $L_1\cup L_2 = \overline{\overline{L_1}\cap\overline{L_2}}$.

In [ ]:
U1 = min_dfa(pruneUnreach(union_dfa(even0, even1)))
U2 = min_dfa(pruneUnreach(comp_dfa(intersect_dfa(comp_dfa(even0), comp_dfa(even1)))))
print("dual law holds? langeq=%s iso=%s" % (langeq_dfa(U1, U2), iso_dfa(U1, U2)))
assert langeq_dfa(U1, U2) and iso_dfa(U1, U2)

So intersection is redundant &mdash; complement and union generate it.

In [ ]:
print("primitive set {comp, union} suffices for the Boolean operations on DFA.")
print("Chapter 10 makes the same argument for regular expressions.")

## 4. Animation

Both sides of DeMorgan's law produce this one minimal machine.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(lhs(even0, even1), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Verify DeMorgan's law on two machines of different sizes.
2. Is `iso_dfa` on minimized machines a **proof**, or only strong evidence? Justify.
3. Which other operations are redundant given complement and union?

In [ ]:
# Your work for the exercises above.